# Model 4: Dense U-Net — Fetal Head Segmentation
**Dataset:** HC18 Grand Challenge — https://zenodo.org/records/1322001

**Architecture:** U-Net with **DenseNet-style dense blocks** in both encoder and decoder. Each layer receives feature maps from ALL preceding layers in that block via concatenation (not just the previous layer). This maximizes feature reuse, improves gradient flow, and produces richer multi-scale representations.

**Why dense connections for ultrasound?** Ultrasound images have weak edges and speckle noise. Dense blocks propagate low-level edge information all the way to deep layers, helping preserve the fetal head boundary information that often gets lost in standard deep architectures.

## 1. Install & Download

In [ ]:
!pip install albumentations opencv-python matplotlib pandas tqdm -q
!wget -q --show-progress 'https://zenodo.org/records/1322001/files/training_set.zip' -O training_set.zip
!wget -q --show-progress 'https://zenodo.org/records/1322001/files/test_set.zip' -O test_set.zip
!unzip -q training_set.zip -d hc18
!unzip -q test_set.zip -d hc18
print('Ready.')

## 2. Imports & Config

In [ ]:
import os, random, math, time, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from glob import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

warnings.filterwarnings('ignore')
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

IMG_SIZE = 256; BATCH_SIZE = 8; EPOCHS = 100; LR = 1e-4; PATIENCE = 15
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'dense_unet'
TRAIN_IMG_DIR = 'hc18/training_set'

print(f'Device: {DEVICE} | Model: Dense U-Net')

## 3. Dataset

In [ ]:
def get_file_pairs(d):
    imgs = sorted(glob(os.path.join(d,'*_HC.png')))
    return [(i, i.replace('_HC.png','_HC_Annotation.png')) for i in imgs if os.path.exists(i.replace('_HC.png','_HC_Annotation.png'))]

def patient_split(pairs, val=0.15, test=0.10):
    rng = random.Random(SEED); p = pairs.copy(); rng.shuffle(p)
    n = len(p); nv, nt = int(n*val), int(n*test)
    return p[nt+nv:], p[nt:nt+nv], p[:nt]

def get_transforms(split):
    if split=='train':
        return A.Compose([A.PadIfNeeded(IMG_SIZE,IMG_SIZE), A.RandomCrop(IMG_SIZE,IMG_SIZE),
            A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
            A.GaussNoise(p=0.3), A.RandomBrightnessContrast(p=0.3),
            A.Normalize(mean=(0.485,0.456,0.406),std=(0.229,0.224,0.225)), ToTensorV2()])
    return A.Compose([A.Resize(IMG_SIZE,IMG_SIZE),
        A.Normalize(mean=(0.485,0.456,0.406),std=(0.229,0.224,0.225)), ToTensorV2()])

class HC18Dataset(Dataset):
    def __init__(self, pairs, tfm=None):
        self.pairs=pairs; self.tfm=tfm
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        ip,mp = self.pairs[idx]
        img=cv2.cvtColor(cv2.imread(ip),cv2.COLOR_BGR2RGB)
        msk=(cv2.imread(mp,cv2.IMREAD_GRAYSCALE)>127).astype(np.float32)
        if self.tfm:
            a=self.tfm(image=img,mask=msk); img,msk=a['image'],a['mask']
        return img, msk.unsqueeze(0)

all_pairs = get_file_pairs(TRAIN_IMG_DIR)
train_pairs, val_pairs, test_pairs = patient_split(all_pairs)
print(f'Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}')

train_dl = DataLoader(HC18Dataset(train_pairs, get_transforms('train')), BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_dl   = DataLoader(HC18Dataset(val_pairs,   get_transforms('val')),   BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_dl  = DataLoader(HC18Dataset(test_pairs,  get_transforms('val')),   BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

## 4. Dense U-Net Architecture

**DenseLayer:** Each layer computes BN → ReLU → Conv3×3 and concatenates its output to the input. If a block has k layers with growth rate g, the total output channels = input_channels + k*g.

**TransitionDown:** 1×1 Conv + MaxPool to halve spatial size while compressing channels.

**TransitionUp:** Bilinear upsample + 1×1 Conv projection.

**Dense U-Net structure:**
```
Input → StemConv → DenseBlock1 → TD → DenseBlock2 → TD → DenseBlock3 → TD → DenseBlock4
      → Bottleneck Dense Block
      → TU → DenseDec3 + skip3 → TU → DenseDec2 + skip2 → TU → DenseDec1 + skip1
      → Head → Sigmoid
```

In [ ]:
class DenseLayer(nn.Module):
    """Single dense layer: BN-ReLU-Conv. Output concatenated to input."""
    def __init__(self, in_ch, growth_rate=32):
        super().__init__()
        self.layer = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, growth_rate, 3, padding=1, bias=False)
        )
    def forward(self, x):
        return torch.cat([x, self.layer(x)], dim=1)


class DenseBlock(nn.Module):
    """N dense layers. Input fed to all layers, all intermediate outputs concatenated."""
    def __init__(self, in_ch, num_layers=4, growth_rate=32):
        super().__init__()
        layers = []
        ch = in_ch
        for _ in range(num_layers):
            layers.append(DenseLayer(ch, growth_rate))
            ch += growth_rate
        self.block = nn.Sequential(*layers)
        self.out_ch = ch

    def forward(self, x): return self.block(x)


class TransitionDown(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.MaxPool2d(2)
        )
    def forward(self, x): return self.net(x)


class TransitionUp(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.proj = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.proj(self.up(x))


class DenseUNet(nn.Module):
    def __init__(self, in_ch=3, growth_rate=32):
        super().__init__()
        g = growth_rate

        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 64, 7, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True)
        )

        # Encoder Dense Blocks
        self.db1  = DenseBlock(64,  num_layers=4, growth_rate=g)   # 64 + 4*32 = 192
        self.td1  = TransitionDown(self.db1.out_ch, 128)

        self.db2  = DenseBlock(128, num_layers=4, growth_rate=g)   # 128+4*32 = 256
        self.td2  = TransitionDown(self.db2.out_ch, 256)

        self.db3  = DenseBlock(256, num_layers=4, growth_rate=g)   # 256+4*32 = 384
        self.td3  = TransitionDown(self.db3.out_ch, 384)

        # Bottleneck
        self.neck = DenseBlock(384, num_layers=4, growth_rate=g)   # 384+4*32 = 512

        # Decoder
        self.tu3  = TransitionUp(self.neck.out_ch, 256)
        self.dec3 = DenseBlock(256 + self.db3.out_ch, num_layers=4, growth_rate=g)

        self.tu2  = TransitionUp(self.dec3.out_ch, 128)
        self.dec2 = DenseBlock(128 + self.db2.out_ch, num_layers=4, growth_rate=g)

        self.tu1  = TransitionUp(self.dec2.out_ch, 64)
        self.dec1 = DenseBlock(64 + self.db1.out_ch, num_layers=4, growth_rate=g)

        self.head = nn.Conv2d(self.dec1.out_ch, 1, 1)

    def forward(self, x):
        x  = self.stem(x)

        s1 = self.db1(x)
        x  = self.td1(s1)

        s2 = self.db2(x)
        x  = self.td2(s2)

        s3 = self.db3(x)
        x  = self.td3(s3)

        x  = self.neck(x)

        x  = self.tu3(x)
        x  = self.dec3(torch.cat([x, s3], dim=1))

        x  = self.tu2(x)
        x  = self.dec2(torch.cat([x, s2], dim=1))

        x  = self.tu1(x)
        x  = self.dec1(torch.cat([x, s1], dim=1))

        return torch.sigmoid(self.head(x))


model = DenseUNet().to(DEVICE)
print(f'Dense U-Net params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 5. Loss, Metrics & Optimizer

In [ ]:
def hybrid_loss(pred, target, smooth=1e-6):
    bce  = F.binary_cross_entropy(pred, target)
    inter = (pred*target).sum(dim=(2,3))
    dice = 1-(2*inter+smooth)/(pred.sum(dim=(2,3))+target.sum(dim=(2,3))+smooth)
    return bce+dice.mean()

def compute_metrics(pred, target, thresh=0.5, smooth=1e-6):
    p=(pred>thresh).float()
    tp=(p*target).sum(); fp=(p*(1-target)).sum()
    fn=((1-p)*target).sum(); tn=((1-p)*(1-target)).sum()
    dice=(2*tp+smooth)/(2*tp+fp+fn+smooth)
    iou=(tp+smooth)/(tp+fp+fn+smooth)
    prec=(tp+smooth)/(tp+fp+smooth); rec=(tp+smooth)/(tp+fn+smooth)
    acc=(tp+tn+smooth)/(tp+tn+fp+fn+smooth)
    spec=(tn+smooth)/(tn+fp+smooth)
    return {'dice':dice.item(),'iou':iou.item(),'precision':prec.item(),'recall':rec.item(),
            'f1':dice.item(),'accuracy':acc.item(),'specificity':spec.item()}

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,'min',patience=5,factor=0.5,verbose=True)
print('Ready.')

## 6. Training

In [ ]:
history = {'train_loss':[],'val_loss':[],'val_dice':[],'val_iou':[]}
best_loss=float('inf'); pat=0

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    tloss=0; all_m={k:0.0 for k in ['dice','iou','precision','recall','f1','accuracy','specificity']}
    ctx=torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs,masks in tqdm(loader, leave=False):
            imgs,masks=imgs.to(DEVICE),masks.to(DEVICE)
            preds=model(imgs); loss=hybrid_loss(preds,masks)
            if train: optimizer.zero_grad(); loss.backward(); optimizer.step()
            tloss+=loss.item()
            m=compute_metrics(preds.detach(),masks)
            for k in all_m: all_m[k]+=m[k]
    n=len(loader)
    return tloss/n,{k:v/n for k,v in all_m.items()}

print(f'Training Dense U-Net for {EPOCHS} epochs...\n')
start=time.time()

for epoch in range(1, EPOCHS+1):
    tl,_=run_epoch(train_dl,True); vl,vm=run_epoch(val_dl,False)
    scheduler.step(vl)
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['val_dice'].append(vm['dice']); history['val_iou'].append(vm['iou'])
    if epoch%10==0 or epoch==1:
        print(f'Ep {epoch:03d} | Train:{tl:.4f} | Val:{vl:.4f} | Dice:{vm["dice"]:.4f} | IoU:{vm["iou"]:.4f}')
    if vl<best_loss: best_loss=vl; torch.save(model.state_dict(),f'best_{MODEL_NAME}.pth'); pat=0
    else:
        pat+=1
        if pat>=PATIENCE: print(f'Early stop at ep {epoch}'); break

print(f'Done in {(time.time()-start)/60:.1f} min')

## 7. Training Curves

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
fig.suptitle('Dense U-Net Training History')
axes[0].plot(history['train_loss'],label='Train'); axes[0].plot(history['val_loss'],label='Val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['val_dice'],color='green'); axes[1].set_title('Val Dice')
axes[2].plot(history['val_iou'],color='orange'); axes[2].set_title('Val IoU')
for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout(); plt.savefig(f'{MODEL_NAME}_curves.png',dpi=150); plt.show()

## 8. Test & HC Estimation

In [ ]:
model.load_state_dict(torch.load(f'best_{MODEL_NAME}.pth', map_location=DEVICE))
test_loss, test_mets = run_epoch(test_dl, False)
print('='*50); print('  Dense U-Net — Test Results'); print('='*50)
for k,v in test_mets.items(): print(f'  {k:<12}: {v*100:.2f}%')
print(f'  {"loss":<12}: {test_loss:.4f}'); print('='*50)

In [ ]:
def estimate_hc(mask_np, ps=0.143):
    u8=(mask_np*255).astype(np.uint8)
    u8=cv2.morphologyEx(u8,cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)))
    cs,_=cv2.findContours(u8,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not cs: return None
    lc=max(cs,key=cv2.contourArea)
    if len(lc)<5: return None
    (_,_),(MA,ma),_=cv2.fitEllipse(lc)
    a,b=MA/2,ma/2
    return math.pi*(3*(a+b)-math.sqrt((3*a+b)*(a+3*b)))*ps

dm=torch.tensor([0.485,0.456,0.406]).view(3,1,1)
ds=torch.tensor([0.229,0.224,0.225]).view(3,1,1)
model.eval(); hc_errors=[]
fig,axes=plt.subplots(3,4,figsize=(16,12))

with torch.no_grad():
    for idx,(imgs,masks) in enumerate(test_dl):
        if idx>=1: break
        preds=model(imgs.to(DEVICE)).cpu()
        for i in range(min(3,imgs.shape[0])):
            inp=(imgs[i]*ds+dm).permute(1,2,0).numpy().clip(0,1)
            mnp=masks[i,0].numpy(); pnp=(preds[i,0]>0.5).numpy().astype(np.float32)
            hcp=estimate_hc(pnp); hcg=estimate_hc(mnp)
            axes[i][0].imshow(inp);              axes[i][0].set_title('Input')
            axes[i][1].imshow(mnp,cmap='gray');  axes[i][1].set_title('GT')
            axes[i][2].imshow(pnp,cmap='gray');  axes[i][2].set_title('Pred')
            ov=(inp*255).astype(np.uint8).copy()
            cs2,_=cv2.findContours((pnp*255).astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(ov,cs2,-1,(0,255,0),2)
            axes[i][3].imshow(ov); axes[i][3].set_title(f'HC={hcp:.1f}mm' if hcp else '—')
            if hcp and hcg: hc_errors.append(abs(hcp-hcg))
            for ax in axes[i]: ax.axis('off')

plt.suptitle('Dense U-Net — Qualitative Results',fontsize=14)
plt.tight_layout(); plt.savefig(f'{MODEL_NAME}_qualitative.png',dpi=150); plt.show()
if hc_errors: print(f'HC MAE: {np.mean(hc_errors):.2f} mm | MSE: {np.mean(np.array(hc_errors)**2):.2f} mm²')

## 9. Save Results

In [ ]:
results={'model':'Dense-UNet',**{k:round(v*100,2) for k,v in test_mets.items()},'test_loss':round(test_loss,4)}
if hc_errors:
    results['hc_mae_mm']=round(float(np.mean(hc_errors)),3)
    results['hc_mse_mm2']=round(float(np.mean(np.array(hc_errors)**2)),3)
df=pd.DataFrame([results])
df.to_csv(f'{MODEL_NAME}_results.csv',index=False)
print(df.to_string(index=False))